In [2]:
import re
from pathlib import Path

import numpy as np
import pandas as pd
from Bio.PDB import PDBParser
from Bio.SeqUtils import seq1

In [3]:
STANDARD_AA = set("ACDEFGHIKLMNPQRSTVWY")
BACKBONE_ATOMS = {"N", "CA", "C", "O"}
EXCLUDED_ATOM_NAMES = {"OXT"}
ORI_CHAIN_ID = "Z"
VALID_ORI_SOURCES = {"l_chain_all_sidechain", "hotspot_sidechain"}

RFD2_ROOT = Path("/home/junjiechen/1_work/250401-Dpepalign/Benchmark/RFdiffusion2")
CSV_PATH = RFD2_ROOT / "alanine-scan" / "ddg_filtered.csv"
MINIMIZED_DIR = RFD2_ROOT / "minimized"
HOTSPOT_OUT_DIR = RFD2_ROOT / "rfd2_hotspots"
MAPPING_LOG_PATH = HOTSPOT_OUT_DIR / "pose_mapping_debug.csv"

parser = PDBParser(QUIET=True)


def residue_uid(chain_id, residue):
    _, resseq, icode = residue.id
    return (chain_id, int(resseq), (icode or " ").strip())


def residue_is_standard_aa(residue):
    hetflag = residue.id[0]
    if hetflag.strip():
        return False
    try:
        aa = seq1(residue.get_resname())
    except Exception:
        return False
    return aa in STANDARD_AA


def atom_is_excluded(atom):
    atom_name = atom.get_name().strip().upper()
    if atom_name in EXCLUDED_ATOM_NAMES:
        return True
    # Covers H, HA, HB2, HG..., and also 1H/2H/3H naming styles.
    if atom_name.startswith("H") or (len(atom_name) > 1 and atom_name[0].isdigit() and atom_name[1] == "H"):
        return True
    return False


def build_pose_index_map(model):
    """Reproduce pose indexing used in gen_mutfile.ipynb."""
    pose_to_residue = {}
    pose_to_meta = {}
    pose_index = 0
    for chain in model:
        for residue in chain:
            if not residue_is_standard_aa(residue):
                continue
            pose_index += 1
            uid = residue_uid(chain.id, residue)
            pose_to_residue[pose_index] = residue
            pose_to_meta[pose_index] = {
                "chain": chain.id,
                "resseq": uid[1],
                "icode": uid[2],
                "resname3": residue.get_resname().strip(),
                "resname1": seq1(residue.get_resname()),
            }
    return pose_to_residue, pose_to_meta


def parse_mutant_series(mutants):
    """Extract wt_aa and pose_index from strict Mutant format like F110A."""
    clean = mutants.astype(str).str.strip()
    m = clean.str.fullmatch(r"([A-Za-z])(\d+)([A-Za-z])")
    if not m.all():
        bad = clean[~m]
        raise ValueError(f"Mutant解析失败(非法格式):\n{bad.to_string(index=False)}")

    parsed = clean.str.extract(r"^([A-Za-z])(\d+)[A-Za-z]$")
    return parsed[0].str.upper(), parsed[1].astype(int)


def get_backbone_atoms_of_chain(model, chain_id="L"):
    if chain_id not in model:
        raise KeyError(f"Chain {chain_id} not found")
    atoms = []
    for residue in model[chain_id]:
        if not residue_is_standard_aa(residue):
            continue
        for atom in residue:
            atom_name = atom.get_name().strip()
            if atom_name in BACKBONE_ATOMS and not atom_is_excluded(atom):
                atoms.append(atom)
    return atoms


def atom_distance(atom1, atom2):
    return float(np.linalg.norm(atom1.coord - atom2.coord))


def min_distance_to_query_atoms(residue, query_atoms):
    residue_atoms = [atom for atom in residue.get_atoms() if not atom_is_excluded(atom)]
    if not residue_atoms:
        return float("inf")

    min_d = float("inf")
    for atom in residue_atoms:
        d = min(atom_distance(atom, q) for q in query_atoms)
        if d < min_d:
            min_d = d
    return min_d


def sidechain_atoms(residue):
    atoms = []
    for atom in residue.get_atoms():
        atom_name = atom.get_name().strip()
        if atom_name in BACKBONE_ATOMS:
            continue
        if atom_is_excluded(atom):
            continue
        atoms.append(atom)
    return atoms


def get_l_chain_all_sidechain_atoms(model, chain_id="L"):
    if chain_id not in model:
        raise KeyError(f"Chain {chain_id} not found")
    atoms = []
    for residue in model[chain_id]:
        if not residue_is_standard_aa(residue):
            continue
        atoms.extend(sidechain_atoms(residue))
    return atoms


def calculate_centroid_from_atoms(atoms):
    if not atoms:
        raise ValueError("No atoms available for centroid calculation")
    coords = np.array([atom.coord for atom in atoms], dtype=float)
    centroid = coords.mean(axis=0)
    return float(centroid[0]), float(centroid[1]), float(centroid[2])


def build_protein_resname_overrides(neighbor_uids, uid_to_residue):
    """Rename non-L extracted residues as single-letter + serial, e.g., W01, F03."""
    overrides = {}
    for i, uid in enumerate(neighbor_uids, start=1):
        residue = uid_to_residue.get(uid)
        if residue is None:
            continue
        aa1 = seq1(residue.get_resname()).upper()
        if aa1 not in STANDARD_AA:
            aa1 = "X"
        overrides[uid] = f"{aa1}{i:02d}"
    return overrides


def format_pdb_atom_line(atom, serial, record="ATOM", resname_override=None):
    res = atom.get_parent()
    chain = res.get_parent()
    atom_name = atom.get_name().strip()
    altloc = atom.get_altloc() if atom.get_altloc() != " " else " "
    if resname_override is None:
        resname = res.get_resname().strip()[:3]
    else:
        resname = str(resname_override).strip()[:3]
    chain_id = chain.id
    _, resseq, icode = res.id
    x, y, z = atom.coord
    occ = atom.get_occupancy() if atom.get_occupancy() is not None else 1.00
    bfac = atom.get_bfactor() if atom.get_bfactor() is not None else 0.00
    element = (atom.element or atom_name[0]).strip().upper()[:2]
    if len(atom_name) < 4:
        atom_name_fmt = f" {atom_name:<3}"
    else:
        atom_name_fmt = atom_name[:4]
    return (
        f"{record:<6}{serial:>5d} {atom_name_fmt}{altloc:1}"
        f"{resname:>3s} {chain_id:1s}{int(resseq):>4d}{(icode or ' '):1s}   "
        f"{x:>8.3f}{y:>8.3f}{z:>8.3f}{occ:>6.2f}{bfac:>6.2f}          {element:>2s}"
    )


def format_ori_hetatm_line(ori_coord, chain_id=ORI_CHAIN_ID):
    x, y, z = ori_coord
    return (
        f"HETATM{9999:>5d} "
        f"{'ORI':^4s} {'ORI':3s} {chain_id:1s}{1:>4d}"
        f"    "
        f"{x:8.3f}{y:8.3f}{z:8.3f}"
        f"{0.00:6.2f}{0.00:6.2f}      "
        f"{'':4s}{'OR':>2s}"
        f"  "
    )


def safe_file_stem(file_name):
    name = Path(str(file_name)).name
    stem = name[4:] if name.startswith("ddg_") else name
    return re.sub(r"[^A-Za-z0-9_.-]", "_", stem)


def process_one_file(file_name, grp_df, l_chain="L", cutoff=8.0, ori_source="l_chain_all_sidechain"):
    try:
        if ori_source not in VALID_ORI_SOURCES:
            raise ValueError(f"Unsupported ori_source: {ori_source}")

        pdb_stem = safe_file_stem(file_name)
        pdb_path = MINIMIZED_DIR / f"{pdb_stem}.pdb"
        if not pdb_path.exists():
            return {
                "file": file_name,
                "status": "missing_pdb",
                "message": str(pdb_path),
                "output_pdb": "",
                "n_l_targets": 0,
                "n_neighbors": 0,
                "n_written_atoms": 0,
                "ori_chain": "",
                "ori_source": ori_source,
                "ori_x": np.nan,
                "ori_y": np.nan,
                "ori_z": np.nan,
            }, []

        structure = parser.get_structure(file_name, str(pdb_path))
        model = structure[0]

        if ORI_CHAIN_ID in {chain.id for chain in model}:
            return {
                "file": file_name,
                "status": "error",
                "message": f"ORI chain conflict: chain {ORI_CHAIN_ID} already exists",
                "output_pdb": "",
                "n_l_targets": 0,
                "n_neighbors": 0,
                "n_written_atoms": 0,
                "ori_chain": "",
                "ori_source": ori_source,
                "ori_x": np.nan,
                "ori_y": np.nan,
                "ori_z": np.nan,
            }, []

        wt_series, pose_series = parse_mutant_series(grp_df["Mutant"])
        grp = grp_df.copy()
        grp["wt_aa"] = wt_series.values
        grp["pose_index"] = pose_series.values

        _, pose_to_meta = build_pose_index_map(model)
        mapping_rows = []
        target_uids = []

        for _, row in grp.iterrows():
            pose_idx = int(row["pose_index"])
            wt_aa = row["wt_aa"]
            mut = row["Mutant"]

            meta = pose_to_meta.get(pose_idx)
            if meta is None:
                mapping_rows.append({
                    "File": file_name,
                    "Mutant": mut,
                    "wt_aa": wt_aa,
                    "pose_index": pose_idx,
                    "mapped": False,
                    "reason": "pose_not_found",
                    "mapped_chain": "",
                    "mapped_resseq": np.nan,
                    "mapped_icode": "",
                    "mapped_resname1": "",
                    "wt_match": False,
                })
                continue

            mapped_chain = meta["chain"]
            mapped_resseq = meta["resseq"]
            mapped_icode = meta["icode"]
            mapped_resname1 = meta["resname1"]
            wt_match = (wt_aa == mapped_resname1)

            reason = "ok"
            if mapped_chain != l_chain:
                reason = "not_in_L_chain"
            elif not wt_match:
                reason = "wt_mismatch"
            else:
                target_uids.append((mapped_chain, mapped_resseq, mapped_icode))

            mapping_rows.append({
                "File": file_name,
                "Mutant": mut,
                "wt_aa": wt_aa,
                "pose_index": pose_idx,
                "mapped": True,
                "reason": reason,
                "mapped_chain": mapped_chain,
                "mapped_resseq": mapped_resseq,
                "mapped_icode": mapped_icode,
                "mapped_resname1": mapped_resname1,
                "wt_match": wt_match,
            })

        target_uids = sorted(set(target_uids), key=lambda x: (x[0], x[1], x[2]))
        if not target_uids:
            return {
                "file": file_name,
                "status": "no_valid_L_targets",
                "message": "No valid L-chain targets after pose mapping",
                "output_pdb": "",
                "n_l_targets": 0,
                "n_neighbors": 0,
                "n_written_atoms": 0,
                "ori_chain": "",
                "ori_source": ori_source,
                "ori_x": np.nan,
                "ori_y": np.nan,
                "ori_z": np.nan,
            }, mapping_rows

        backbone_atoms = get_backbone_atoms_of_chain(model, chain_id=l_chain)
        if not backbone_atoms:
            return {
                "file": file_name,
                "status": "no_L_backbone",
                "message": "L chain has no standard backbone atoms",
                "output_pdb": "",
                "n_l_targets": len(target_uids),
                "n_neighbors": 0,
                "n_written_atoms": 0,
                "ori_chain": "",
                "ori_source": ori_source,
                "ori_x": np.nan,
                "ori_y": np.nan,
                "ori_z": np.nan,
            }, mapping_rows

        uid_to_residue = {
            residue_uid(chain.id, residue): residue
            for chain in model
            for residue in chain
            if residue_is_standard_aa(residue)
        }

        neighbor_uids = []
        for chain in model:
            if chain.id == l_chain:
                continue
            for residue in chain:
                if not residue_is_standard_aa(residue):
                    continue
                dmin = min_distance_to_query_atoms(residue, backbone_atoms)
                if dmin < cutoff:
                    neighbor_uids.append((chain.id, residue.id[1], (residue.id[2] or " ").strip()))

        neighbor_uids = sorted(set(neighbor_uids), key=lambda x: (x[0], x[1], x[2]))

        if ori_source == "l_chain_all_sidechain":
            ori_atoms = get_l_chain_all_sidechain_atoms(model, chain_id=l_chain)
        else:
            ori_atoms = []
            for uid in target_uids:
                res = uid_to_residue.get(uid)
                if res is None:
                    continue
                ori_atoms.extend(sidechain_atoms(res))
        ori_coord = calculate_centroid_from_atoms(ori_atoms)

        protein_resname_overrides = build_protein_resname_overrides(neighbor_uids, uid_to_residue)

        HOTSPOT_OUT_DIR.mkdir(parents=True, exist_ok=True)
        out_path = HOTSPOT_OUT_DIR / f"{Path(str(file_name)).name}_hotspot.pdb"

        serial = 1
        written = 0
        with open(out_path, "w") as f:
            # Keep original residue names for L-chain hotspot residues.
            for uid in target_uids:
                res = uid_to_residue.get(uid)
                if res is None:
                    continue
                for atom in sidechain_atoms(res):
                    f.write(format_pdb_atom_line(atom, serial, record="ATOM") + "\n")
                    serial += 1
                    written += 1

            # Rename extracted protein-chain residues as W01/F03/A03...
            for uid in neighbor_uids:
                res = uid_to_residue.get(uid)
                if res is None:
                    continue
                resname_override = protein_resname_overrides.get(uid)
                for atom in sidechain_atoms(res):
                    f.write(
                        format_pdb_atom_line(
                            atom, serial, record="ATOM", resname_override=resname_override
                        ) + "\n"
                    )
                    serial += 1
                    written += 1

            f.write(format_ori_hetatm_line(ori_coord, chain_id=ORI_CHAIN_ID) + "\n")
            f.write("END\n")

        status = "ok" if written > 0 else "empty_hotspot"
        message = "" if written > 0 else "No sidechain atoms written"
        return {
            "file": file_name,
            "status": status,
            "message": message,
            "output_pdb": str(out_path),
            "n_l_targets": len(target_uids),
            "n_neighbors": len(neighbor_uids),
            "n_written_atoms": written,
            "ori_chain": ORI_CHAIN_ID,
            "ori_source": ori_source,
            "ori_x": ori_coord[0],
            "ori_y": ori_coord[1],
            "ori_z": ori_coord[2],
        }, mapping_rows
    except Exception as exc:
        return {
            "file": file_name,
            "status": "error",
            "message": f"{type(exc).__name__}: {exc}",
            "output_pdb": "",
            "n_l_targets": 0,
            "n_neighbors": 0,
            "n_written_atoms": 0,
            "ori_chain": "",
            "ori_source": ori_source,
            "ori_x": np.nan,
            "ori_y": np.nan,
            "ori_z": np.nan,
        }, []


def validate_input_df(df):
    required = {"File", "Mutant"}
    missing = sorted(required.difference(df.columns))
    if missing:
        raise ValueError(f"CSV缺少必要列: {missing}")

    mask_valid = df["File"].notna() & df["Mutant"].notna()
    if not mask_valid.all():
        bad_n = int((~mask_valid).sum())
        print(f"警告: 跳过 {bad_n} 行 File/Mutant 缺失的数据")
        df = df.loc[mask_valid].copy()

    df["File"] = df["File"].astype(str).str.strip()
    df["Mutant"] = df["Mutant"].astype(str).str.strip()

    mask_nonempty = (df["File"] != "") & (df["Mutant"] != "")
    if not mask_nonempty.all():
        bad_n = int((~mask_nonempty).sum())
        print(f"警告: 跳过 {bad_n} 行 File/Mutant 为空字符串的数据")
        df = df.loc[mask_nonempty].copy()

    return df


def run_stage1_stage2_debug(
    csv_path=CSV_PATH,
    smoke_n=3,
    run_full=False,
    l_chain="L",
    cutoff=8.0,
    ori_source="l_chain_all_sidechain",
    write_mapping_log=True,
):
    if ori_source not in VALID_ORI_SOURCES:
        raise ValueError(
            f"ori_source必须是 {sorted(VALID_ORI_SOURCES)} 之一, 当前: {ori_source}"
        )

    df_raw = pd.read_csv(csv_path)
    df = validate_input_df(df_raw)

    grouped = list(df.groupby("File", sort=True))
    selected = grouped if run_full else grouped[:smoke_n]

    print(f"Total files in CSV: {len(grouped)}")
    print(f"Files to process now: {len(selected)}")
    print(f"ORI source: {ori_source}")

    summaries = []
    mapping_all = []
    for file_name, grp in selected:
        summary, mapping_rows = process_one_file(
            file_name, grp, l_chain=l_chain, cutoff=cutoff, ori_source=ori_source
        )
        summaries.append(summary)
        mapping_all.extend(mapping_rows)

        msg = (
            f"[{summary['status']}] {file_name} | "
            f"L_targets={summary['n_l_targets']} neighbors={summary['n_neighbors']} atoms={summary['n_written_atoms']}"
        )
        if summary["ori_chain"]:
            msg += f" ORI=({summary['ori_x']:.3f}, {summary['ori_y']:.3f}, {summary['ori_z']:.3f})"
        print(msg)

    df_summary = pd.DataFrame(summaries)
    df_mapping = pd.DataFrame(mapping_all)

    if write_mapping_log and not df_mapping.empty:
        HOTSPOT_OUT_DIR.mkdir(parents=True, exist_ok=True)
        df_mapping.to_csv(MAPPING_LOG_PATH, index=False)
        print(f"\nMapping log saved: {MAPPING_LOG_PATH}")

    return df_summary, df_mapping


# Debug run: set run_full=True when smoke test passes
df_debug_summary, df_debug_mapping = run_stage1_stage2_debug(
    smoke_n=3,
    run_full=True,
    l_chain="L",
    cutoff=8.0,
    ori_source="l_chain_all_sidechain",
    write_mapping_log=True,
 )

display(df_debug_summary)
display(df_debug_mapping.head(20))

Total files in CSV: 148
Files to process now: 148
ORI source: l_chain_all_sidechain
[ok] ddg_1ddv_0012 | L_targets=1 neighbors=19 atoms=90 ORI=(12.254, 18.412, -0.591)
[ok] ddg_1eg4_0003 | L_targets=2 neighbors=25 atoms=124 ORI=(53.750, 16.599, 35.935)
[ok] ddg_1f47_0001 | L_targets=6 neighbors=19 atoms=103 ORI=(9.338, 11.238, 14.377)
[ok] ddg_1f8h_0003 | L_targets=2 neighbors=21 atoms=95 ORI=(3.895, -0.621, -1.155)
[ok] ddg_1j2x_0001 | L_targets=6 neighbors=21 atoms=110 ORI=(-0.946, -2.193, 9.972)
[ok] ddg_1jd5_0001 | L_targets=3 neighbors=30 atoms=146 ORI=(-0.708, 47.139, 12.938)
[ok] ddg_1jq8_0001 | L_targets=2 neighbors=31 atoms=140 ORI=(13.131, 21.543, 42.654)
[ok] ddg_1jw6_0019 | L_targets=2 neighbors=19 atoms=94 ORI=(12.546, 31.709, 25.323)
[ok] ddg_1l2z_0001 | L_targets=1 neighbors=17 atoms=98 ORI=(12.182, 0.460, 1.292)
[ok] ddg_1lb6_0004 | L_targets=3 neighbors=26 atoms=131 ORI=(12.313, 23.066, 10.739)
[ok] ddg_1mv0_0001 | L_targets=3 neighbors=30 atoms=153 ORI=(-0.969, -11.44

,file,status,message,output_pdb,n_l_targets,n_neighbors,n_written_atoms,ori_chain,ori_source,ori_x,ori_y,ori_z
0,ddg_1ddv_0012,ok,,/home/junjiechen/1_work/250401-Dpepalign/Bench...,1,19,90,Z,l_chain_all_sidechain,12.253952,18.412381,-0.591190
1,ddg_1eg4_0003,ok,,/home/junjiechen/1_work/250401-Dpepalign/Bench...,2,25,124,Z,l_chain_all_sidechain,53.749833,16.598593,35.934945
2,ddg_1f47_0001,ok,,/home/junjiechen/1_work/250401-Dpepalign/Bench...,6,19,103,Z,l_chain_all_sidechain,9.338446,11.238014,14.376946
3,ddg_1f8h_0003,ok,,/home/junjiechen/1_work/250401-Dpepalign/Bench...,2,21,95,Z,l_chain_all_sidechain,3.895462,-0.620769,-1.154577
4,ddg_1j2x_0001,ok,,/home/junjiechen/1_work/250401-Dpepalign/Bench...,6,21,110,Z,l_chain_all_sidechain,-0.945857,-2.192730,9.971873
...,...,...,...,...,...,...,...,...,...,...,...,...
143,ddg_6f0y_0011,ok,,/home/junjiechen/1_work/250401-Dpepalign/Bench...,5,24,120,Z,l_chain_all_sidechain,-10.440464,-6.354768,-9.017911
144,ddg_6f6d_0002,ok,,/home/junjiechen/1_work/250401-Dpepalign/Bench...,6,68,302,Z,l_chain_all_sidechain,96.270732,-19.331857,22.406536
145,ddg_6fbk_0001,ok,,/home/junjiechen/1_work/250401-Dpepalign/Bench...,5,18,107,Z,l_chain_all_sidechain,37.878873,33.148841,4.399222
146,ddg_6g5g_0004,ok,,/home/junjiechen/1_work/250401-Dpepalign/Bench...,6,27,165,Z,l_chain_all_sidechain,20.902349,-4.031205,9.425940


,File,Mutant,wt_aa,pose_index,mapped,reason,mapped_chain,mapped_resseq,mapped_icode,mapped_resname1,wt_match
0,ddg_1ddv_0012,F110A,F,110,True,ok,L,1006,,F,True
1,ddg_1eg4_0003,Y271A,Y,271,True,ok,L,12,,Y,True
2,ddg_1eg4_0003,R266A,R,266,True,ok,L,7,,R,True
3,ddg_1f47_0001,F155A,F,155,True,ok,L,11,,F,True
4,ddg_1f47_0001,Y149A,Y,149,True,ok,L,5,,Y,True
5,ddg_1f47_0001,L156A,L,156,True,ok,L,12,,L,True
6,ddg_1f47_0001,I152A,I,152,True,ok,L,8,,I,True
7,ddg_1f47_0001,L150A,L,150,True,ok,L,6,,L,True
8,ddg_1f47_0001,D148A,D,148,True,ok,L,4,,D,True
9,ddg_1f8h_0003,F100A,F,100,True,ok,L,111,,F,True


In [4]:
def residue_all_atoms(residue):
    """Return all residue atoms except excluded atoms (H/OXT)."""
    atoms = []
    for atom in residue.get_atoms():
        if atom_is_excluded(atom):
            continue
        atoms.append(atom)
    return atoms


def build_chain_position_index(model):
    """Map standard-AA residue uid to its sequential position within each chain."""
    chain_position_index = {}
    for chain in model:
        pos = 0
        for residue in chain:
            if not residue_is_standard_aa(residue):
                continue
            pos += 1
            chain_position_index[residue_uid(chain.id, residue)] = pos
    return chain_position_index


def split_continuous_neighbor_fragments(neighbor_uids, chain_position_index):
    """Split neighbor residues into continuous fragments using chain-local order."""
    if not neighbor_uids:
        return []

    fragments = []
    current = [neighbor_uids[0]]

    for uid in neighbor_uids[1:]:
        prev = current[-1]
        same_chain = uid[0] == prev[0]
        prev_pos = chain_position_index.get(prev)
        curr_pos = chain_position_index.get(uid)
        if same_chain and prev_pos is not None and curr_pos is not None and curr_pos == prev_pos + 1:
            current.append(uid)
        else:
            fragments.append(current)
            current = [uid]

    fragments.append(current)
    return fragments


def build_protein_resname_overrides(neighbor_uids, uid_to_residue, chain_position_index):
    """
    Naming rule:
    - Continuous fragment (len >= 2): all residues share Uxx (U01..U26).
    - Discontinuous singletons: old naming aa+index, e.g. W01, F02.
    """
    fragments = split_continuous_neighbor_fragments(neighbor_uids, chain_position_index)
    if len(fragments) > 26:
        raise ValueError(
            f"fragment_count_exceeds_26: discontinuous_fragments={len(fragments)}"
        )

    overrides = {}
    u_counter = 0
    single_counter = 0

    for fragment in fragments:
        if len(fragment) >= 2:
            u_counter += 1
            fragment_name = f"U{u_counter:02d}"
            for uid in fragment:
                overrides[uid] = fragment_name
        else:
            uid = fragment[0]
            residue = uid_to_residue.get(uid)
            if residue is None:
                continue
            aa1 = seq1(residue.get_resname()).upper()
            if aa1 not in STANDARD_AA:
                aa1 = "X"
            single_counter += 1
            overrides[uid] = f"{aa1}{single_counter:02d}"

    return overrides, len(fragments), fragments


def format_pdb_atom_line(atom, serial, record="ATOM", resname_override=None):
    if resname_override is not None:
        record = "HETATM"
    res = atom.get_parent()
    chain = res.get_parent()
    atom_name = atom.get_name().strip()
    altloc = atom.get_altloc() if atom.get_altloc() != " " else " "
    if resname_override is None:
        resname = res.get_resname().strip()[:3]
    else:
        resname = str(resname_override).strip()[:3]
    chain_id = chain.id
    _, resseq, icode = res.id
    x, y, z = atom.coord
    occ = atom.get_occupancy() if atom.get_occupancy() is not None else 1.00
    bfac = atom.get_bfactor() if atom.get_bfactor() is not None else 0.00
    element = (atom.element or atom_name[0]).strip().upper()[:2]
    if len(atom_name) < 4:
        atom_name_fmt = f" {atom_name:<3}"
    else:
        atom_name_fmt = atom_name[:4]
    return (
        f"{record:<6}{serial:>5d} {atom_name_fmt}{altloc:1}"
        f"{resname:>3s} {chain_id:1s}{int(resseq):>4d}{(icode or ' '):1s}   "
        f"{x:>8.3f}{y:>8.3f}{z:>8.3f}{occ:>6.2f}{bfac:>6.2f}          {element:>2s}"
    )


def process_one_file(file_name, grp_df, l_chain="L", cutoff=8.0, ori_source="l_chain_all_sidechain"):
    try:
        if ori_source not in VALID_ORI_SOURCES:
            raise ValueError(f"Unsupported ori_source: {ori_source}")

        pdb_stem = safe_file_stem(file_name)
        pdb_path = MINIMIZED_DIR / f"{pdb_stem}.pdb"
        if not pdb_path.exists():
            return {
                "file": file_name,
                "status": "missing_pdb",
                "message": str(pdb_path),
                "output_pdb": "",
                "n_l_targets": 0,
                "n_neighbors": 0,
                "n_fragments": 0,
                "n_written_atoms": 0,
                "ori_chain": "",
                "ori_source": ori_source,
                "ori_x": np.nan,
                "ori_y": np.nan,
                "ori_z": np.nan,
            }, []

        structure = parser.get_structure(file_name, str(pdb_path))
        model = structure[0]

        if ORI_CHAIN_ID in {chain.id for chain in model}:
            return {
                "file": file_name,
                "status": "error",
                "message": f"ORI chain conflict: chain {ORI_CHAIN_ID} already exists",
                "output_pdb": "",
                "n_l_targets": 0,
                "n_neighbors": 0,
                "n_fragments": 0,
                "n_written_atoms": 0,
                "ori_chain": "",
                "ori_source": ori_source,
                "ori_x": np.nan,
                "ori_y": np.nan,
                "ori_z": np.nan,
            }, []

        wt_series, pose_series = parse_mutant_series(grp_df["Mutant"])
        grp = grp_df.copy()
        grp["wt_aa"] = wt_series.values
        grp["pose_index"] = pose_series.values

        _, pose_to_meta = build_pose_index_map(model)
        mapping_rows = []
        target_uids = []

        for _, row in grp.iterrows():
            pose_idx = int(row["pose_index"])
            wt_aa = row["wt_aa"]
            mut = row["Mutant"]

            meta = pose_to_meta.get(pose_idx)
            if meta is None:
                mapping_rows.append({
                    "File": file_name,
                    "Mutant": mut,
                    "wt_aa": wt_aa,
                    "pose_index": pose_idx,
                    "mapped": False,
                    "reason": "pose_not_found",
                    "mapped_chain": "",
                    "mapped_resseq": np.nan,
                    "mapped_icode": "",
                    "mapped_resname1": "",
                    "wt_match": False,
                })
                continue

            mapped_chain = meta["chain"]
            mapped_resseq = meta["resseq"]
            mapped_icode = meta["icode"]
            mapped_resname1 = meta["resname1"]
            wt_match = wt_aa == mapped_resname1

            reason = "ok"
            if mapped_chain != l_chain:
                reason = "not_in_L_chain"
            elif not wt_match:
                reason = "wt_mismatch"
            else:
                target_uids.append((mapped_chain, mapped_resseq, mapped_icode))

            mapping_rows.append({
                "File": file_name,
                "Mutant": mut,
                "wt_aa": wt_aa,
                "pose_index": pose_idx,
                "mapped": True,
                "reason": reason,
                "mapped_chain": mapped_chain,
                "mapped_resseq": mapped_resseq,
                "mapped_icode": mapped_icode,
                "mapped_resname1": mapped_resname1,
                "wt_match": wt_match,
            })

        target_uids = sorted(set(target_uids), key=lambda x: (x[0], x[1], x[2]))
        if not target_uids:
            return {
                "file": file_name,
                "status": "no_valid_L_targets",
                "message": "No valid L-chain targets after pose mapping",
                "output_pdb": "",
                "n_l_targets": 0,
                "n_neighbors": 0,
                "n_fragments": 0,
                "n_written_atoms": 0,
                "ori_chain": "",
                "ori_source": ori_source,
                "ori_x": np.nan,
                "ori_y": np.nan,
                "ori_z": np.nan,
            }, mapping_rows

        backbone_atoms = get_backbone_atoms_of_chain(model, chain_id=l_chain)
        if not backbone_atoms:
            return {
                "file": file_name,
                "status": "no_L_backbone",
                "message": "L chain has no standard backbone atoms",
                "output_pdb": "",
                "n_l_targets": len(target_uids),
                "n_neighbors": 0,
                "n_fragments": 0,
                "n_written_atoms": 0,
                "ori_chain": "",
                "ori_source": ori_source,
                "ori_x": np.nan,
                "ori_y": np.nan,
                "ori_z": np.nan,
            }, mapping_rows

        uid_to_residue = {
            residue_uid(chain.id, residue): residue
            for chain in model
            for residue in chain
            if residue_is_standard_aa(residue)
        }
        chain_position_index = build_chain_position_index(model)

        neighbor_uids = []
        for chain in model:
            if chain.id == l_chain:
                continue
            for residue in chain:
                if not residue_is_standard_aa(residue):
                    continue
                dmin = min_distance_to_query_atoms(residue, backbone_atoms)
                if dmin < cutoff:
                    neighbor_uids.append((chain.id, residue.id[1], (residue.id[2] or " ").strip()))

        neighbor_uids = sorted(set(neighbor_uids), key=lambda x: (x[0], x[1], x[2]))

        try:
            protein_resname_overrides, n_fragments, _fragments = build_protein_resname_overrides(
                neighbor_uids, uid_to_residue, chain_position_index
            )
        except ValueError as exc:
            return {
                "file": file_name,
                "status": "error",
                "message": str(exc),
                "output_pdb": "",
                "n_l_targets": len(target_uids),
                "n_neighbors": len(neighbor_uids),
                "n_fragments": len(split_continuous_neighbor_fragments(neighbor_uids, chain_position_index)),
                "n_written_atoms": 0,
                "ori_chain": "",
                "ori_source": ori_source,
                "ori_x": np.nan,
                "ori_y": np.nan,
                "ori_z": np.nan,
            }, mapping_rows

        if ori_source == "l_chain_all_sidechain":
            ori_atoms = get_l_chain_all_sidechain_atoms(model, chain_id=l_chain)
        else:
            ori_atoms = []
            for uid in target_uids:
                res = uid_to_residue.get(uid)
                if res is None:
                    continue
                # L-chain requirement: only sidechain atoms of hotspot residues.
                ori_atoms.extend(sidechain_atoms(res))
        ori_coord = calculate_centroid_from_atoms(ori_atoms)

        HOTSPOT_OUT_DIR.mkdir(parents=True, exist_ok=True)
        out_path = HOTSPOT_OUT_DIR / f"{Path(str(file_name)).name}_hotspot.pdb"

        serial = 1
        written = 0
        with open(out_path, "w") as f:
            # L-chain keeps hotspot sidechain atoms only.
            for uid in target_uids:
                res = uid_to_residue.get(uid)
                if res is None:
                    continue
                for atom in sidechain_atoms(res):
                    f.write(format_pdb_atom_line(atom, serial, record="ATOM") + "\n")
                    serial += 1
                    written += 1

            # Non-L extracted residues: write full residue atoms as HETATM.
            for uid in neighbor_uids:
                res = uid_to_residue.get(uid)
                if res is None:
                    continue
                resname_override = protein_resname_overrides.get(uid)
                for atom in residue_all_atoms(res):
                    f.write(
                        format_pdb_atom_line(
                            atom, serial, record="ATOM", resname_override=resname_override
                        ) + "\n"
                    )
                    serial += 1
                    written += 1

            f.write(format_ori_hetatm_line(ori_coord, chain_id=ORI_CHAIN_ID) + "\n")
            f.write("END\n")

        status = "ok" if written > 0 else "empty_hotspot"
        message = "" if written > 0 else "No atoms written"
        return {
            "file": file_name,
            "status": status,
            "message": message,
            "output_pdb": str(out_path),
            "n_l_targets": len(target_uids),
            "n_neighbors": len(neighbor_uids),
            "n_fragments": n_fragments,
            "n_written_atoms": written,
            "ori_chain": ORI_CHAIN_ID,
            "ori_source": ori_source,
            "ori_x": ori_coord[0],
            "ori_y": ori_coord[1],
            "ori_z": ori_coord[2],
        }, mapping_rows
    except Exception as exc:
        return {
            "file": file_name,
            "status": "error",
            "message": f"{type(exc).__name__}: {exc}",
            "output_pdb": "",
            "n_l_targets": 0,
            "n_neighbors": 0,
            "n_fragments": 0,
            "n_written_atoms": 0,
            "ori_chain": "",
            "ori_source": ori_source,
            "ori_x": np.nan,
            "ori_y": np.nan,
            "ori_z": np.nan,
        }, []


df_debug_summary, df_debug_mapping = run_stage1_stage2_debug(
    smoke_n=3,
    run_full=True,
    l_chain="L",
    cutoff=8.0,
    ori_source="l_chain_all_sidechain",
    write_mapping_log=True,
)

display(df_debug_summary)
display(df_debug_mapping.head(20))

Total files in CSV: 148
Files to process now: 148
ORI source: l_chain_all_sidechain
[ok] ddg_1ddv_0012 | L_targets=1 neighbors=19 atoms=166 ORI=(12.254, 18.412, -0.591)
[ok] ddg_1eg4_0003 | L_targets=2 neighbors=25 atoms=224 ORI=(53.750, 16.599, 35.935)
[ok] ddg_1f47_0001 | L_targets=6 neighbors=19 atoms=179 ORI=(9.338, 11.238, 14.377)
[ok] ddg_1f8h_0003 | L_targets=2 neighbors=21 atoms=179 ORI=(3.895, -0.621, -1.155)
[ok] ddg_1j2x_0001 | L_targets=6 neighbors=21 atoms=194 ORI=(-0.946, -2.193, 9.972)
[ok] ddg_1jd5_0001 | L_targets=3 neighbors=30 atoms=266 ORI=(-0.708, 47.139, 12.938)
[ok] ddg_1jq8_0001 | L_targets=2 neighbors=31 atoms=264 ORI=(13.131, 21.543, 42.654)
[ok] ddg_1jw6_0019 | L_targets=2 neighbors=19 atoms=170 ORI=(12.546, 31.709, 25.323)
[ok] ddg_1l2z_0001 | L_targets=1 neighbors=17 atoms=166 ORI=(12.182, 0.460, 1.292)
[ok] ddg_1lb6_0004 | L_targets=3 neighbors=26 atoms=235 ORI=(12.313, 23.066, 10.739)
[ok] ddg_1mv0_0001 | L_targets=3 neighbors=30 atoms=273 ORI=(-0.969, -1

,file,status,message,output_pdb,n_l_targets,n_neighbors,n_fragments,n_written_atoms,ori_chain,ori_source,ori_x,ori_y,ori_z
0,ddg_1ddv_0012,ok,,/home/junjiechen/1_work/250401-Dpepalign/Bench...,1,19,7,166,Z,l_chain_all_sidechain,12.253952,18.412381,-0.591190
1,ddg_1eg4_0003,ok,,/home/junjiechen/1_work/250401-Dpepalign/Bench...,2,25,7,224,Z,l_chain_all_sidechain,53.749833,16.598593,35.934945
2,ddg_1f47_0001,ok,,/home/junjiechen/1_work/250401-Dpepalign/Bench...,6,19,12,179,Z,l_chain_all_sidechain,9.338446,11.238014,14.376946
3,ddg_1f8h_0003,ok,,/home/junjiechen/1_work/250401-Dpepalign/Bench...,2,21,7,179,Z,l_chain_all_sidechain,3.895462,-0.620769,-1.154577
4,ddg_1j2x_0001,ok,,/home/junjiechen/1_work/250401-Dpepalign/Bench...,6,21,8,194,Z,l_chain_all_sidechain,-0.945857,-2.192730,9.971873
...,...,...,...,...,...,...,...,...,...,...,...,...,...
143,ddg_6f0y_0011,ok,,/home/junjiechen/1_work/250401-Dpepalign/Bench...,5,24,3,216,Z,l_chain_all_sidechain,-10.440464,-6.354768,-9.017911
144,ddg_6f6d_0002,ok,,/home/junjiechen/1_work/250401-Dpepalign/Bench...,6,68,21,574,Z,l_chain_all_sidechain,96.270732,-19.331857,22.406536
145,ddg_6fbk_0001,ok,,/home/junjiechen/1_work/250401-Dpepalign/Bench...,5,18,6,179,Z,l_chain_all_sidechain,37.878873,33.148841,4.399222
146,ddg_6g5g_0004,ok,,/home/junjiechen/1_work/250401-Dpepalign/Bench...,6,27,11,273,Z,l_chain_all_sidechain,20.902349,-4.031205,9.425940


,File,Mutant,wt_aa,pose_index,mapped,reason,mapped_chain,mapped_resseq,mapped_icode,mapped_resname1,wt_match
0,ddg_1ddv_0012,F110A,F,110,True,ok,L,1006,,F,True
1,ddg_1eg4_0003,Y271A,Y,271,True,ok,L,12,,Y,True
2,ddg_1eg4_0003,R266A,R,266,True,ok,L,7,,R,True
3,ddg_1f47_0001,F155A,F,155,True,ok,L,11,,F,True
4,ddg_1f47_0001,Y149A,Y,149,True,ok,L,5,,Y,True
5,ddg_1f47_0001,L156A,L,156,True,ok,L,12,,L,True
6,ddg_1f47_0001,I152A,I,152,True,ok,L,8,,I,True
7,ddg_1f47_0001,L150A,L,150,True,ok,L,6,,L,True
8,ddg_1f47_0001,D148A,D,148,True,ok,L,4,,D,True
9,ddg_1f8h_0003,F100A,F,100,True,ok,L,111,,F,True
